# Analysis: Student Music Listening and Study Habits

### Rename Features

In [91]:
import pandas as pd

df = pd.read_csv('stats_responses.csv')

columns = df.columns.tolist()

df = df.rename(columns={
    columns[0]: 'timestamp',
    columns[1]: 'agreement',
    columns[2]: 'major',
    # Section 2: Music Listening Habits
    columns[3]: 'pop',
    columns[4]: 'hiphop',
    columns[5]: 'rock',
    columns[6]: 'instrumental',
    columns[7]: 'classical',
    columns[8]: 'Metal',
    columns[9]: 'lofi',
    columns[10]: 'country',
    columns[11]: 'indie',
    columns[12]: 'kpop',
    columns[13]: 'edm',
    columns[14]: 'ambient',
    columns[15]: 'study_music_genre',
    columns[16]: 'leisure_music_genre',
    columns[17]: 'listen_comments',
    # Section 3: Study Habits
    columns[18]: 'study_long_without_breaks',
    columns[19]: 'study_easily_distracted',
    columns[20]: 'study_spaced_evenly',
    columns[21]: 'study_uses_detailed_schedule',
    columns[22]: 'study_prefers_group',
    columns[23]: 'study_prefers_silence',
    columns[24]: 'study_prefers_background_audio',
    columns[25]: 'study_consistent_time',
    columns[26]: 'study_mood_dependent',
    columns[27]: 'study_sustained_motivation',
    columns[28]: 'study_index',
    columns[29]: 'study_comments'
}, errors='raise')

print(df.columns.tolist())

['timestamp', 'agreement', 'major', 'pop', 'hiphop', 'rock', 'instrumental', 'classical', 'Metal', 'lofi', 'country', 'indie', 'kpop', 'edm', 'ambient', 'study_music_genre', 'leisure_music_genre', 'listen_comments', 'study_long_without_breaks', 'study_easily_distracted', 'study_spaced_evenly', 'study_uses_detailed_schedule', 'study_prefers_group', 'study_prefers_silence', 'study_prefers_background_audio', 'study_consistent_time', 'study_mood_dependent', 'study_sustained_motivation', 'study_index', 'study_comments']


### Clean 'major' Feature

In [92]:
original_major = df['major'].copy()

In [93]:
from rapidfuzz import process, fuzz

# 1. Normalize – lowercase, strip whitespace, remove periods/punctuation
df['major'] = (
    original_major
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r'[^\w\s]', '', regex=True)
)

# 2. Mapping abbreviations and known variations
mapping = {
    'cs': 'computer science',
    'math': 'mathematics'
}

df['major'] = df['major'].replace(mapping)

# 3. Fuzzy matching for typos
# Defining "source of truth" list of valid majors
official_majors = [
    'actuarial science',
    'computer science', 
    'data science',
    'computer engineering',
    'electrical engineering',
    'mechanical engineering',
    'biological systems engineering',
    'business analytics',
    'business management',
    'accounting',
    'finance',
    'economics',
    'advertising',
    'neuroscience'
]

# Function to snap typos to the closest official major
def standardize_major(val, choices, threshold=85):
    if pd.isna(val) or val in choices:
        return val
    
    # Find the best match
    match, score, _ = process.extractOne(val, choices, scorer=fuzz.WRatio)
    
    # Only replace if the similarity score is above the threshold (0-100)
    return match if score >= threshold else val

# Apply ONLY to unique values first (much faster than applying to the whole column)
unique_vals = df['major'].unique()
fuzzy_map = {val: standardize_major(val, official_majors) for val in unique_vals}

df['major'] = df['major'].map(fuzzy_map)

# 4. Audit the changes
# View a side-by-side comparison of everything that was modified
# Combine original and cleaned into a temporary audit table
audit_df = pd.DataFrame({
    'before': original_major,
    'after': df['major']
})

# Filter for rows where the value actually changed (ignoring simple case/whitespace differences if desired)
changed = audit_df[audit_df['before'].str.lower().str.strip() != audit_df['after']]

# Group by unique transformations and count how many rows each affected
audit_summary = (
    changed
    .value_counts(dropna=False)
    .reset_index(name='row_count')
)

print(audit_summary.to_string(index=False))
print(df['major'].value_counts().head(50))

                                          before                  after  row_count
                   Finance and Computer Science        computer science          1
                      Economics and Data Science           data science          1
                                              CS       computer science          1
                         Neuroscience/psychology           neuroscience          1
                           Ccomputer Engineering   computer engineering          1
Electrical Engineering and Robotics Engineering  electrical engineering          1
                 Computer Science & Data Science       computer science          1
                  Advertising & Public Relations            advertising          1
major
computer science                  22
actuarial science                  7
computer engineering               7
mechanical engineering             5
data science                       4
electrical engineering             4
business analytics       

### Plotting 'major' Feature

### Clean 'study_music_genre' and 'leisure_music_genre' Feature

### Plotting Music Listening Habits

In [99]:
genre_cols = [
    'pop', 
    'hiphop', 
    'rock', 
    'instrumental', 
    'classical', 
    'Metal', 
    'lofi', 
    'country', 
    'indie', 
    'kpop', 
    'edm', 
    'ambient'
]

mapping = {
    'NEVER or N/A': 0,
    'ONLY when STUDYING': 1,
    'when NOT STUDYING': 2,
    'BOTH when studying and not studying': 3
}

df[genre_cols] = df[genre_cols].replace(mapping)

print(df.head())
print(df[study_cols].isna().sum())
print(df[study_cols].dtypes)


             timestamp     agreement              major pop hiphop rock  \
0  10/30/2026 13:40:57  I Understand  actuarial science   3      2    3   
1  10/30/2026 13:41:09  I Understand  actuarial science   3      1    0   
2  10/30/2026 13:41:12  I Understand       data science   2      2    2   
3  10/30/2026 13:41:21  I Understand            finance   2      3    0   
4  10/30/2026 13:41:22  I Understand   computer science   3      2    3   

  instrumental classical Metal lofi  ... study_spaced_evenly  \
0            0         0     0    0  ...                  -1   
1            1         1     0    0  ...                   1   
2            1         1     0    0  ...                   2   
3            1         1     0    1  ...                   0   
4            3         1     2    1  ...                   0   

  study_uses_detailed_schedule study_prefers_group study_prefers_silence  \
0                           -1                   0                    -1   
1           

### Plotting Study Habits

In [101]:
print(df.columns.tolist())
study_cols = [
    'study_long_without_breaks', 
    'study_easily_distracted', 
    'study_spaced_evenly', 
    'study_uses_detailed_schedule', 
    'study_prefers_group', 
    'study_prefers_silence', 
    'study_prefers_background_audio', 
    'study_consistent_time', 
    'study_mood_dependent', 
    'study_sustained_motivation'
]

mapping = {
    'strongly disagree': -2, '-2': -2,
    'disagree': -1,          '-1': -1,
    'neutral': 0,            '0': 0,
    'agree': 1,              '1': 1,
    'strongly agree': 2,     '2': 2
}


# Clean whitespace/casing and map each column safely
for col in study_cols:
    df[col] = df[col].astype(str).str.strip().str.lower().map(mapping)
print(df.head())
print(df[study_cols].isna().sum())
print(df[study_cols].dtypes)

['timestamp', 'agreement', 'major', 'pop', 'hiphop', 'rock', 'instrumental', 'classical', 'Metal', 'lofi', 'country', 'indie', 'kpop', 'edm', 'ambient', 'study_music_genre', 'leisure_music_genre', 'listen_comments', 'study_long_without_breaks', 'study_easily_distracted', 'study_spaced_evenly', 'study_uses_detailed_schedule', 'study_prefers_group', 'study_prefers_silence', 'study_prefers_background_audio', 'study_consistent_time', 'study_mood_dependent', 'study_sustained_motivation', 'study_index', 'study_comments']
             timestamp     agreement              major pop hiphop rock  \
0  10/30/2026 13:40:57  I Understand  actuarial science   3      2    3   
1  10/30/2026 13:41:09  I Understand  actuarial science   3      1    0   
2  10/30/2026 13:41:12  I Understand       data science   2      2    2   
3  10/30/2026 13:41:21  I Understand            finance   2      3    0   
4  10/30/2026 13:41:22  I Understand   computer science   3      2    3   

  instrumental classical Met